In [2]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["kopi_nusantara"]

# Memilih (atau otomatis membuat) collection
koleksi_menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'local', 'praktikum_bigdata']


In [9]:
# Menghapus data lama jika ada (agar percobaan kita mulai dari kondisi bersih)
koleksi_menu.delete_many({})

# insert_many() — menyisipkan banyak dokumen sekaligus (12 dokumen)
koleksi_menu.insert_many([
    {"nama_menu": "Kopi Susu", "kategori": "Kopi", "harga": 25000, "stok": 50, "tingkat_kepahitan": 1, "tags": ["panas", "susu enak"]},
    {"nama_menu": "Cafe Latte", "kategori": "Kopi", "harga": 30000, "stok": 50, "tingkat_kepahitan": 2, "tags": ["panas"]},
    {"nama_menu": "Cappucino", "kategori": "Kopi", "harga": 28000, "stok": 50, "tingkat_kepahitan": 3, "tags": ["panas"]},
    {"nama_menu": "Americano Dingin", "kategori": "Kopi", "harga": 28000, "stok": 50, "tingkat_kepahitan": 4, "tags": ["dingin", "pahit"]},
    {"nama_menu": "Expresso", "kategori": "Kopi", "harga": 30000, "stok": 50, "tingkat_kepahitan": 5, "tags": ["kuat", "pahit"]},
    {"nama_menu": "Kopi Hitam", "kategori": "Kopi", "harga": 18000, "stok": 50, "tingkat_kepahitan": 4, "tags": ["kuat", "melek"]},
    {"nama_menu": "Mie Goreng Nusantara", "kategori": "Makanan", "harga": 15000, "stok": 50, "pedas_level": 2, "tags": ["enak"]},
    {"nama_menu": "Nasi Goreng", "kategori": "Makanan", "harga": 15000, "stok": 100, "pedas_level": 3, "tags": ["gurih", "nasi"]},
    {"nama_menu": "Mie Goreng Sambal Matah", "kategori": "Makanan", "harga": 15000, "stok": 60, "pedas_level": 3, "tags": ["pedas", "matah"]},
    {"nama_menu": "Pisang Coklat", "kategori": "Makanan", "harga": 12000, "stok": 30, "pedas_level": 0, "tags": ["gurih", "manis"]},
    {"nama_menu": "Dimsum Ayam", "kategori": "Makanan", "harga": 20000, "stok": 25, "pedas_level": 2, "tags": ["hangat", "kukus"]},
    {"nama_menu": "Ayam Geprek", "kategori": "Makanan", "harga": 15000, "stok": 60, "pedas_level": 3, "tags": ["gurih", "pedas"]}
])

print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))

Jumlah dokumen di collection 'menu': 12


In [12]:
# find() — mengambil SEMUA dokumen yang cocok (mengembalikan cursor, di-looping atau di-list())
# Seluruh menu kategori Kopi dengan tingkat_kepahitan di atas 3 
print("Seluruh menu kategori Kopi dengan tingkat_kepahitan di atas 3 :")
for p in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {p['nama_menu']} (Kepahitan: {p['tingkat_kepahitan']}, Rp{p['harga']:,})")

print()

# Seluruh menu dengan harga di antara 15000 dan 30000 ($gte: 15000 dan $lte: 30000)
print("Menu dengan harga di antara 15000 dan 30000:")
for p in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {p['nama_menu']} (Rp{p['harga']:,})")

print()

# Menu yang tidak memiliki field pedas_level sama sekali ($exists: False)
print("Menu yang tidak memiliki field pedas_level:")
for p in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {p['nama_menu']} (Kategori: {p['kategori']})")

Seluruh menu kategori Kopi dengan tingkat_kepahitan di atas 3 :
  - Americano Dingin (Kepahitan: 4, Rp28,000)
  - Expresso (Kepahitan: 5, Rp30,000)
  - Kopi Hitam (Kepahitan: 4, Rp18,000)

Menu dengan harga di antara 15000 dan 30000:
  - Kopi Susu (Rp25,000)
  - Cafe Latte (Rp30,000)
  - Cappucino (Rp28,000)
  - Americano Dingin (Rp28,000)
  - Expresso (Rp30,000)
  - Kopi Hitam (Rp18,000)
  - Mie Goreng Nusantara (Rp15,000)
  - Nasi Goreng (Rp15,000)
  - Mie Goreng Sambal Matah (Rp15,000)
  - Dimsum Ayam (Rp20,000)
  - Ayam Geprek (Rp15,000)

Menu yang tidak memiliki field pedas_level:
  - Kopi Susu (Kategori: Kopi)
  - Cafe Latte (Kategori: Kopi)
  - Cappucino (Kategori: Kopi)
  - Americano Dingin (Kategori: Kopi)
  - Expresso (Kategori: Kopi)
  - Kopi Hitam (Kategori: Kopi)


In [23]:
# menambahkan field status "Tersedia" ke seluruh dokumen
koleksi_menu.update_many(
    {},  # filter kosong untuk semua dokumen
    {"$set": {"status": "Tersedia"}}
)

# update_one() — mengubah status menjadi "Habis" khusus untuk menu Expresso
koleksi_menu.update_one(
    {"nama_menu": "Expresso"},
    {"$set": {"status": "Habis"}}
)

# update_one() — mengubah status menjadi "Habis" khusus untuk menu Ayam Geprek
koleksi_menu.update_one(
    {"nama_menu": "Ayam Geprek"},
    {"$set": {"status": "Habis"}}
)

print("Setelah update status:")
for p in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {p['nama_menu']}: status = {p['status']}")

Setelah update status:
  - Kopi Susu: status = Tersedia
  - Cafe Latte: status = Tersedia
  - Cappucino: status = Tersedia
  - Americano Dingin: status = Tersedia
  - Expresso: status = Habis
  - Kopi Hitam: status = Tersedia
  - Mie Goreng Nusantara: status = Tersedia
  - Nasi Goreng: status = Tersedia
  - Mie Goreng Sambal Matah: status = Tersedia
  - Pisang Coklat: status = Tersedia
  - Dimsum Ayam: status = Tersedia
  - Ayam Geprek: status = Habis


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_menu.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# menampilkan printSchema() hasil konversi
print("1. printSchema() dari Spark DataFrame:")
df_spark_menu.printSchema()

print()

# Rata-rata harga per kategori menggunakan groupBy
print("2. Rata-rata harga per kategori:")
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

/home/imam/anaconda3/envs/bigdata/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


1. printSchema() dari Spark DataFrame:
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- tags: string (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)


2. Rata-rata harga per kategori:


[Stage 0:>                                                        (0 + 12) / 12]

+--------+------------------+-----------+
|kategori|   rata_rata_harga|jumlah_menu|
+--------+------------------+-----------+
|    Kopi|           26500.0|          6|
| Makanan|15333.333333333334|          6|
+--------+------------------+-----------+



Tantangan yang saya temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam) adalah adanya ketidakseragaman struktur field. Contohnya di bagian menu kopi tidak memiliki field pedas_level dan kemudian di menu makanan tidak memiliki field tingkat_kepahitan. Nah, hal ini menyebabkan masalah, karena Spark DataFrame menuntut adanya skema tabel yang kaku dan seragam untuk seluruh baris datanya.

Nah, untuk mengatasi tantang tersebut diperlukan Pandas DataFrame sebagai perantara (jembatan) sebelum dikonversi ke Spark DataFrame. Saat data ditarik dari MongoDB ke Pandas, baris dokumen yang tidak memiliki field tertentu akan secara otomatis diisi dengan nilai kosong. Kemudian ketika data Pandas ini dikonversi ke Spark DataFrame, Spark akan mendeteksi nilai kosong tersebut dan menetapkannya/mengisinya menjadi nilai null. Dengan cara ini, perbedaan struktur antar dokumen akan dapat ditangani secara otomatis, sehingga Spark tetap berhasil membentuk skema tabel yang seragam dan rapi tanpa menimbulkan error. 

In [ ]:
# menutup spark session
spark.stop()
print("SparkSession ditutup.")